In [ ]:
# from CSVtoSQL import CSVFolderToSQL
# CSVFolderToSQL("../synthea/output/csv", "medical_data.db").csv_to_sql()

In [1]:
import pandas as pd
import numpy as np
from SQL_query import SQLQuery
SQL = SQLQuery("medical_data.db")
query = """
WITH LATEST_PREGNANCY AS (
    SELECT
        PATIENT,
        MAX(DATE(START)) AS PREGNANCY_START,
        DATE(MAX(START), '+270 days') AS EST_DELIVERY_DATE
    FROM encounters
    WHERE DESCRIPTION = 'Prenatal initial visit (regime/therapy)'
    GROUP BY PATIENT
),
LATEST_PREG_DIA AS (
    SELECT
        o.PATIENT,
        MAX(DATE(o.DATE)) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Diastolic Blood Pressure' THEN o.VALUE END) AS Diastolic_Blood_Pressure
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Diastolic Blood Pressure' AND DATE(o.DATE) <= lp.EST_DELIVERY_DATE 
    GROUP BY o.PATIENT
),
LATEST_PREG_SYS AS (
    SELECT
        o.PATIENT,
        MAX(DATE(o.DATE)) AS OBSERVATION_DATE,
        MAX(CASE WHEN o.DESCRIPTION = 'Systolic Blood Pressure' THEN o.VALUE END) AS Systolic_Blood_Pressure
    FROM observations as o
    INNER JOIN LATEST_PREGNANCY as lp ON o.PATIENT = lp.PATIENT
    WHERE o.DESCRIPTION = 'Systolic Blood Pressure' AND DATE(o.DATE) <= lp.EST_DELIVERY_DATE
    GROUP BY o.PATIENT
),
DEATH_CAUSE AS (
    SELECT
        d.PATIENT,
        d.VALUE AS CAUSE_OF_DEATH
    FROM observations as d
    WHERE d.DESCRIPTION = 'Cause of Death [US Standard Certificate of Death]'
)
SELECT 
    t1.Id,
    DATE(t1.BIRTHDATE) AS BIRTHDATE,
    DATE(t1.DEATHDATE) AS DEATHDATE,
    t1.RACE,
    t1.ETHNICITY,
    t1.GENDER,
    t1.COUNTY,
    t1.HEALTHCARE_EXPENSES,
    t1.HEALTHCARE_COVERAGE,
    t1.INCOME,
    t1.HEALTHCARE_EXPENSES - t1.INCOME AS RELATIVE_EXPENSES,
    DATE(t2.PREGNANCY_START) AS PREGNANCY_START,
    DATE(t2.EST_DELIVERY_DATE) AS EST_DELIVERY_DATE,
    DATE(t3.OBSERVATION_DATE) AS BP_OBSERVATION_DATE,
    t3.Diastolic_Blood_Pressure,
    t4.Systolic_Blood_Pressure,
    t8.CAUSE_OF_DEATH
    
FROM patients as t1
INNER JOIN LATEST_PREGNANCY as t2 ON t1.Id = t2.PATIENT
LEFT JOIN LATEST_PREG_DIA as t3 ON t1.Id = t3.PATIENT
LEFT JOIN LATEST_PREG_SYS as t4 ON t1.Id = t4.PATIENT
LEFT JOIN DEATH_CAUSE as t8 ON t1.Id = t8.PATIENT
"""
LAfemMed = SQL.execute_query(query)
LAfemMed['BIRTHDATE'] = pd.to_datetime(LAfemMed['BIRTHDATE'])
LAfemMed['DEATHDATE'] = pd.to_datetime(LAfemMed['DEATHDATE'])
LAfemMed['PREGNANCY_START'] = pd.to_datetime(LAfemMed['PREGNANCY_START'])
LAfemMed['EST_DELIVERY_DATE'] = pd.to_datetime(LAfemMed['EST_DELIVERY_DATE'])
LAfemMed['BP_OBSERVATION_DATE'] = pd.to_datetime(LAfemMed['BP_OBSERVATION_DATE'])
LAfemMed['Days_Pregancy_to_Death'] = (LAfemMed['DEATHDATE'] - LAfemMed['PREGNANCY_START']).dt.days
LAfemMed['Maternal_Mortality'] = np.where(LAfemMed['Days_Pregancy_to_Death'] <= 270+42, 1, 0)

Maternal_Deaths = LAfemMed[LAfemMed['Maternal_Mortality'] == 1]

# Preprocess Data

In [2]:
from sklearn.preprocessing import StandardScaler
numeric_cols = ['INCOME', 'RELATIVE_EXPENSES', 'Diastolic_Blood_Pressure', 'Systolic_Blood_Pressure']
all_cols = ['INCOME', 'RELATIVE_EXPENSES', 'Diastolic_Blood_Pressure', 'Systolic_Blood_Pressure', 'Maternal_Mortality']
final_data = LAfemMed[all_cols].dropna()
scale = StandardScaler()
X = scale.fit_transform(final_data[numeric_cols])
y = final_data['Maternal_Mortality'].to_numpy().astype(float)

In [ ]:
from sklearn.naive_bayes import GaussianNB
GNB = GaussianNB()
GNB.fit(X,y)
pred = GNB.predict(X)

In [ ]:
from LogisticRegression import LR_class
LR = LR_class()
LR_fit = LR.LogRes(X,y)
LR_fit.predict(X)

In [3]:
from SVM import SVM_class
svm = SVM_class()
svm.best_fit(X,y)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan]
  warnings.warn(


{'best': SVC(C=0.6, gamma=0.1, random_state=42),
 'model': RandomizedSearchCV(cv=5, estimator=SVC(random_state=42), n_iter=20, n_jobs=8,
                    param_distributions={'C': array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]),
                                         'gamma': [0.01, 0.05, 0.1, 0.5]},
                    scoring='neg_log_loss', verbose=2)}

In [ ]:
import RandForest
rf = RandForest.RF_hyperparams()
rf.best_fit(X,y)

In [ ]:
from NNmod import NN_mod
nn_mort = NN_mod(epochs=1000)
nn_mort.fit(X, y)

In [ ]:
from XGBoost import XGBclass
xgb = XGBclass(X, y)
best_params = xgb.xgb_Kfold(k=5)
best_params